# P3.09 Discovery: Dataset Read/Write Probe

**PURPOSE:** Test Kaggle dataset I/O, versioning, and artifact creation capabilities

**TIMEOUT:** ≤5 minutes

**CRITICAL:** P3.09 workers need to read assets from datasets and write shard outputs.

In [ ]:
import json
import os
import time
from datetime import datetime
from pathlib import Path

DISCOVERY_SESSION = {
    "session_id": f"dataset_probe_{int(time.time())}",
    "timestamp": datetime.now().isoformat(),
    "timeout_hard": 300,
    "timeout_warning": 270,
    "notebook_name": "kaggle_discovery_07_dataset_readwrite_probe",
    "results": []
}

start_time = time.time()

def log_test(test_name, result, evidence, duration_s):
    DISCOVERY_SESSION["results"].append({
        "test": test_name,
        "result": result,
        "evidence": evidence,
        "duration_s": duration_s,
        "timestamp": datetime.now().isoformat()
    })
    print(f"[{result:8s}] {test_name} ({duration_s:.1f}s)")

def check_timeout():
    elapsed = time.time() - start_time
    if elapsed > DISCOVERY_SESSION["timeout_hard"]:
        raise RuntimeError(f"HARD TIMEOUT: {elapsed:.0f}s")
    return elapsed

print(f"🔬 P3.09 Dataset Read/Write Probe: {DISCOVERY_SESSION['session_id']}")
print(f"📍 Started: {DISCOVERY_SESSION['timestamp']}")

## Test 1: /kaggle/input/ Read Access

In [ ]:
test_start = time.time()
check_timeout()

try:
    input_probe = {
        "input_path_exists": False,
        "is_directory": False,
        "readable": False,
        "contents_sample": []
    }
    
    input_dir = Path("/kaggle/input")
    input_probe["input_path_exists"] = input_dir.exists()
    
    if input_probe["input_path_exists"]:
        input_probe["is_directory"] = input_dir.is_dir()
        try:
            # Try to list contents
            contents = list(input_dir.iterdir())
            input_probe["readable"] = True
            input_probe["file_count"] = len(contents)
            # Sample first few items
            input_probe["contents_sample"] = [str(c.name) for c in contents[:5]]
        except PermissionError:
            input_probe["readable"] = False
    
    result_status = "PASS" if input_probe["readable"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("input_directory_read", result_status, input_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("input_directory_read", "FAIL", str(e)[:100], duration)

## Test 2: /kaggle/working/ Write Access

In [ ]:
test_start = time.time()
check_timeout()

try:
    working_probe = {
        "working_path_exists": False,
        "writable": False,
        "test_file_written": False,
        "test_file_readable": False,
        "space_available_mb": 0
    }
    
    working_dir = Path("/kaggle/working")
    working_probe["working_path_exists"] = working_dir.exists()
    
    if working_probe["working_path_exists"]:
        # Try to write a test file
        test_file = working_dir / "test_write.txt"
        try:
            test_file.write_text("test content")
            working_probe["test_file_written"] = True
            working_probe["writable"] = True
            
            # Try to read it back
            content = test_file.read_text()
            working_probe["test_file_readable"] = content == "test content"
            
            # Clean up
            test_file.unlink()
        except Exception as e:
            working_probe["write_error"] = str(e)[:50]
        
        # Check available space
        try:
            stat = os.statvfs("/kaggle/working")
            free_bytes = stat.f_bavail * stat.f_frsize
            working_probe["space_available_mb"] = int(free_bytes / 1024 / 1024)
        except:
            pass
    
    result_status = "PASS" if working_probe["writable"] else "FAIL"
    duration = time.time() - test_start
    log_test("working_directory_write", result_status, working_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("working_directory_write", "FAIL", str(e)[:100], duration)

## Test 3: Dataset Versioning Capability

In [ ]:
test_start = time.time()
check_timeout()

try:
    version_probe = {
        "kaggle_api_available": False,
        "dataset_api_accessible": False,
        "versioning_note": "Kaggle datasets support versioning; each push creates a new version"
    }
    
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        version_probe["kaggle_api_available"] = True
        
        # Try to access dataset API (may fail without authentication)
        try:
            api = KaggleApi()
            # Check if we have dataset methods available
            version_probe["dataset_api_accessible"] = hasattr(api, "dataset_create_version")
        except:
            pass
    except ImportError:
        pass
    
    result_status = "PASS" if version_probe["kaggle_api_available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("dataset_versioning", result_status, version_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("dataset_versioning", "FAIL", str(e)[:100], duration)

## Test 4: Output Artifact Creation Proof

In [ ]:
test_start = time.time()
check_timeout()

try:
    artifact_probe = {
        "test_artifact_created": False,
        "artifact_size_bytes": 0,
        "artifact_path": None
    }
    
    # Create a test artifact (simulating shard output)
    artifact_data = {
        "shard_id": "test_shard_0",
        "frame_range": {"start": 0, "end": 100},
        "render_time_s": 42.5,
        "output_file": "test_shard_0.mp4",
        "hash": "abc123def456",
        "timestamp": datetime.now().isoformat()
    }
    
    working_dir = Path("/kaggle/working")
    artifact_file = working_dir / "test_artifact.json"
    
    try:
        artifact_file.write_text(json.dumps(artifact_data, indent=2))
        artifact_probe["test_artifact_created"] = artifact_file.exists()
        artifact_probe["artifact_size_bytes"] = artifact_file.stat().st_size
        artifact_probe["artifact_path"] = str(artifact_file)
        
        # Clean up
        artifact_file.unlink()
    except Exception as e:
        artifact_probe["error"] = str(e)[:50]
    
    result_status = "PASS" if artifact_probe["test_artifact_created"] else "FAIL"
    duration = time.time() - test_start
    log_test("output_artifact_creation", result_status, artifact_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("output_artifact_creation", "FAIL", str(e)[:100], duration)

## Final Report

In [ ]:
results = DISCOVERY_SESSION["results"]
passed = sum(1 for r in results if r["result"] == "PASS")
failed = sum(1 for r in results if r["result"] == "FAIL")
unknown = sum(1 for r in results if r["result"] == "UNKNOWN")

total_time = time.time() - start_time

DISCOVERY_SESSION.update({
    "summary": {
        "total_tests": len(results),
        "passed": passed,
        "failed": failed,
        "unknown": unknown,
        "total_time_s": total_time,
        "verdict": "Dataset I/O validated" if passed >= 2 else "Dataset I/O needs investigation"
    }
})

output_path = Path("/kaggle/working/discovery_dataset_probe_results.json")
output_path.write_text(json.dumps(DISCOVERY_SESSION, indent=2))

print(f"\n📊 Dataset Read/Write Summary:")
print(f"   PASS:    {passed}")
print(f"   FAIL:    {failed}")
print(f"   UNKNOWN: {unknown}")
print(f"   Total:   {len(results)} tests in {total_time:.1f}s")
print(f"\n✅ Results saved to: {output_path}")